# Clinical Risk Classification (MIMIC-IV)
This notebook uses clinical vital signs (Heart Rate, Blood Pressure, Glucose, etc.) to predict if a patient is at high clinical risk (e.g., Sepsis).

### Cell 1: Setup
Initialization of data loaders and Scikit-Learn tools.

In [ ]:
%run ../data/Data_Management.ipynb
import pandas as pd, numpy as np, joblib, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
loader = DataLoader()

### Cell 2: Feature Selection
We load the MIMIC synthetic dataset and select 13 critical clinical features. `clean_mimic_data` handles missing value imputation and clinical range validation.

In [ ]:
df_raw = loader.load_mimic_file("sepsis_icu_synthetic.csv")
if df_raw is not None:
    df_c = clean_mimic_data(df_raw)
    features = ["age", "hr_mean", "sbp_mean", "dbp_mean", "map_mean", "temp_celsius_mean", "spo2_mean", "respiratory_rate_mean", "wbc", "lactate_mmol", "glucose", "sofa_score", "qsofa"]
    X, y = df_c[[f for f in features if f in df_c.columns]], df_c["sepsis_label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Cell 3: Training
Training a Random Forest classifier to identify critical vs. non-critical clinical states.

In [ ]:
risk_clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
risk_clf.fit(X_train, y_train)
print(classification_report(y_test, risk_clf.predict(X_test)))
joblib.dump(risk_clf, "../../models/risk_classification_v1.joblib")